#Phase 3: Granular Aspect Extraction and Data Decomposition

##3.1: Targeted Scanning

In [17]:
import pandas as pd
import os
import re
from nltk.tokenize import sent_tokenize

aspect_dictionary = {
    "Product Quality": [
        "coffee", "matcha", "latte", "espresso", "sikwate", "barako", "pandesal",
        "pastry", "cake", "taste", "lami", "masarap", "lasa", "tam-is", "umay",
        "lamiay", "food", "drinks", "lasaw", "tab-ang", "init", "bugnaw"
    ],
    "Customer Service": [
        "barista", "staff", "crew", "cashier", "service", "mabait", "accommodating",
        "paspas", "dugay", "maayos", "attitude", "buotan", "snob", "friendly"
    ],
    "Ambiance and Atmosphere": [
        "ambiance", "vibe", "aesthetic", "chill", "quiet", "noisy", "saba", "music",
        "muni-muni", "chika", "tropa", "atmosphere", "crowded", "ig-worthy",
        "instagrammable", "tambay"
    ],
    "Facilities and Amenities": [
        "wifi", "internet", "connection", "socket", "outlet", "plug", "cr", "restroom",
        "comfort room", "aircon", "tables", "chairs", "space", "study", "tuon", "work",
        "laptop", "kusog", "igang"
    ],
    "Price and Value": [
        "price", "value", "mahal", "barato", "sulit", "tipid", "overpriced",
        "affordable", "sayang", "budget", "expensive", "cheap", "pricy"
    ],
    "Store Operations and Accessibility": [
        "parking", "location", "accessible", "layo", "malayo", "duol", "hours", "open",
        "close", "traffic"
    ]
}

In [21]:
base_dir = 'D:/Ateneo de Davao/Thesis-Project-Aspect-Based-Sentiment-Analysis/Dataset'
file_path = os.path.join(base_dir, 'Thesis_master_reviews_nonull.csv')

df = pd.read_csv(file_path)

In [22]:
df

,place_name,latitude,longitude,rating,review_text,shop_type,clean_review
0,"10,000 Reasons Cafe",7.067546,125.613287,5,This is easily one of my favorite spots! The f...,coffee,This is easily one of my favorite spots! The f...
1,"10,000 Reasons Cafe",7.067546,125.613287,5,"There are some good reasons to go here, maybe ...",coffee,"There are some good reasons to go here, maybe ..."
2,"10,000 Reasons Cafe",7.067546,125.613287,5,The staffs are welcoming and the atmosphere is...,coffee,The staffs are welcoming and the atmosphere is...
3,"10,000 Reasons Cafe",7.067546,125.613287,5,Seriously loved this cafe! It's so cute and pe...,coffee,Seriously loved this cafe! It's so cute and pe...
4,"10,000 Reasons Cafe",7.067546,125.613287,5,"This place is a really good spot for studying,...",coffee,"This place is a really good spot for studying,..."
...,...,...,...,...,...,...,...
12566,Whisk It Good Matcha,7.088042,125.594479,5,Delicious food; accommodating staffs; and the ...,matcha,Delicious food; accommodating staffs; and the ...
12567,Whisk It Good Matcha,7.088042,125.594479,5,Matcha is delicious.,matcha,Matcha is delicious.
12568,Yoro No Matcha x Crumb,7.057481,125.487751,4,"Matcha and Sea Salt Hojicha are the best, but ...",matcha,"Matcha and Sea Salt Hojicha are the best, but ..."
12569,Yoro No Matcha x Crumb,7.057481,125.487751,5,Afforadable and legit matcha in Toril,matcha,Afforadable and legit matcha in Toril


In [23]:
#Phase 3.1: Targeted Scanning for Aspect Extraction

# prevent substring mismatches
compiled_patterns = {
    aspect: re.compile(r'\b(?:' + '|'.join(map(re.escape, keywords)) + r')\b')
    for aspect, keywords in aspect_dictionary.items()
}

def target_scan(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    return [aspect for aspect, pattern in compiled_patterns.items() if pattern.search(text)]

# 'df' and text is in 'clean_review'
# 1. Tokenize reviews into sentences (with strict string validation to bypass rogue floats/NaNs)
df['sentence'] = df['clean_review'].apply(lambda x: sent_tokenize(x) if isinstance(x, str) else [])

# 2. Explode the sentences into distinct rows
df_sentences = df.explode('sentence').reset_index(drop=True)

# 3. Execute Step 3.1: Targeted Scanning
df_sentences['TargetAspect'] = df_sentences['sentence'].apply(target_scan)

# 4. Clean up: Drop sentences where no aspect was detected
df_scanned = df_sentences[df_sentences['TargetAspect'].astype(bool)].copy()

#3.2: Review-Aspect Pair Isolation

In [24]:
# Step 3.2: Data Explosion (Isolating Review-Aspect Pairs)
df_exploded = df_scanned.explode('TargetAspect').reset_index(drop=True)

In [25]:
df 

,place_name,latitude,longitude,rating,review_text,shop_type,clean_review,sentence
0,"10,000 Reasons Cafe",7.067546,125.613287,5,This is easily one of my favorite spots! The f...,coffee,This is easily one of my favorite spots! The f...,"[This is easily one of my favorite spots!, The..."
1,"10,000 Reasons Cafe",7.067546,125.613287,5,"There are some good reasons to go here, maybe ...",coffee,"There are some good reasons to go here, maybe ...","[There are some good reasons to go here, maybe..."
2,"10,000 Reasons Cafe",7.067546,125.613287,5,The staffs are welcoming and the atmosphere is...,coffee,The staffs are welcoming and the atmosphere is...,[The staffs are welcoming and the atmosphere i...
3,"10,000 Reasons Cafe",7.067546,125.613287,5,Seriously loved this cafe! It's so cute and pe...,coffee,Seriously loved this cafe! It's so cute and pe...,"[Seriously loved this cafe!, It's so cute and ..."
4,"10,000 Reasons Cafe",7.067546,125.613287,5,"This place is a really good spot for studying,...",coffee,"This place is a really good spot for studying,...",[This place is a really good spot for studying...
...,...,...,...,...,...,...,...,...
12566,Whisk It Good Matcha,7.088042,125.594479,5,Delicious food; accommodating staffs; and the ...,matcha,Delicious food; accommodating staffs; and the ...,[Delicious food; accommodating staffs; and the...
12567,Whisk It Good Matcha,7.088042,125.594479,5,Matcha is delicious.,matcha,Matcha is delicious.,[Matcha is delicious.]
12568,Yoro No Matcha x Crumb,7.057481,125.487751,4,"Matcha and Sea Salt Hojicha are the best, but ...",matcha,"Matcha and Sea Salt Hojicha are the best, but ...","[Matcha and Sea Salt Hojicha are the best, but..."
12569,Yoro No Matcha x Crumb,7.057481,125.487751,5,Afforadable and legit matcha in Toril,matcha,Afforadable and legit matcha in Toril,[Afforadable and legit matcha in Toril]


In [26]:
df_exploded.to_csv('Exploded_Review_Aspect_Pairs.csv', index=False, encoding='utf-8')     

# Output the class distribution for Phase 5 Threshold Evaluation
print(df_exploded['TargetAspect'].value_counts())

TargetAspect
Product Quality                       11040
Customer Service                       4611
Ambiance and Atmosphere                3085
Facilities and Amenities               1919
Store Operations and Accessibility     1208
Price and Value                        1195
Name: count, dtype: int64
